# Building a synthetic InSAR scene with exact ground truth

*An undergraduate-level walkthrough of `parvaneh.synthetic`.*

---

### Why would anybody invent an interferogram?

A phase-unwrapping algorithm takes a **wrapped** phase image -- where every
value has been squeezed into the interval $[-\pi, \pi]$ -- and tries to
recover the **continuous** phase that produced it. To say whether an algorithm
is any good, you have to know the right answer. Real interferograms do not come
with one: nobody was standing on the volcano with a ruler.

So we build the scene ourselves, backwards:

1. choose a **ground displacement** field from a closed-form elastic model;
2. **project** that displacement onto the satellite's line of sight;
3. convert displacement into **phase** with an exact formula;
4. add **coherence** and physically correct **noise**;
5. **wrap** the phase -- and, crucially, record the integer number of cycles
   that wrapping destroyed.

Because steps 1-4 are analytic, every quantity an unwrapper is supposed to
recover is known exactly. That is the whole point.

### What you need to know first

* Phase is an angle, measured in **radians**. A full cycle is $2\pi$ radians.
* A **fringe** -- one full colour cycle in an interferogram -- corresponds to
  **half a wavelength** of ground movement along the line of sight. For
  Sentinel-1 that is $\lambda/2 \approx 2.8$ cm. This is the punchline of
  radar interferometry: centimetre motions are visible from 700 km up.
* Nothing here is an image-processing trick. Every number below comes from an
  equation you could derive on paper.

Let us begin.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Make the repository importable when the notebook is run from notebooks/.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
# The import package lives in src/ (see pyproject.toml); an installed
# copy would be found on sys.path anyway, so this is a no-op then.
SRC = ROOT / "src"
if (SRC / "parvaneh").is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import parvaneh.synthetic as syn
from parvaneh import SAKURA, SAKURA_DIVERGING

TWO_PI = 2.0 * np.pi
print("numpy", np.__version__)
print("synthetic package exports:", len(syn.__all__), "names")


## 1. The ground grid

Everything lives on a regular East-North-Up grid. Two conventions matter and
are worth memorising now, because they cause more bugs than any equation:

* **Row 0 is the northernmost row**, so $y$ *decreases* as the row index
  grows. This matches `imshow(origin="upper")`, so images plot without flips.
* **Column 0 is the westernmost column**; $x$ increases with the column index.

`Grid.centered()` puts the mean of the pixel *centres* exactly at
$(0, 0)$ -- a convenience for symmetric sources such as Mogi.

We use 256 x 256 pixels at 40 m spacing, i.e. roughly 10 km on a side. That is
a small but realistic volcano-monitoring frame.

In [ ]:
grid = syn.Grid.centered(nx=256, ny=256, spacing=40.0)
print(grid.describe())
print()
print("shape (ny, nx)      :", grid.shape)
print("X[0, 0], Y[0, 0]    :", (grid.X[0, 0], grid.Y[0, 0]), " <- north-west corner")
print("X[-1, -1], Y[-1, -1]:", (grid.X[-1, -1], grid.Y[-1, -1]), " <- south-east corner")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), constrained_layout=True)
for ax, field, name in zip(axes, (grid.X, grid.Y), ("X (east) [m]", "Y (north) [m]")):
    im = ax.imshow(field / 1e3, origin="upper", extent=grid.extent, cmap="viridis")
    ax.set_title(name)
    ax.set_xlabel("Easting [km]")
    ax.set_ylabel("Northing [km]")
    fig.colorbar(im, ax=ax, shrink=0.85)
plt.show()


## 2. Two deformation sources, both analytic

### 2a. Mogi: a pressurised sphere (magma chamber)

Mogi (1958) solved the elastic half-space for a small pressurised sphere. In the
form used throughout this package,

$$u = \frac{(1-\nu)\,\Delta V}{\pi R^{3}}\;(\Delta x,\; \Delta y,\; d), \qquad R^{2} = \Delta x^{2} + \Delta y^{2} + d^{2}$$

with $\nu$ the Poisson ratio, $\Delta V$ the volume change in m$^{3}$ and $d$
the source depth. Directly above the source the ground rises by
$(1-\nu)\Delta V / (\pi d^{2})$, so a shallower or larger source raises the
surface more. Two things to notice, and one trap:

* **Symmetry.** The horizontal field points radially outward (inflation) or
  inward (deflation); directly above the source it is exactly zero.
* **Decay.** Far away the vertical term behaves like $d/R^{3}\propto r^{-3}$.
  Doubling the distance divides the uplift by roughly **eight**, not four.

> **Trap.** Different textbooks define "volume change" with different factors of
> $3/(4\pi)$ folded in, so absolute amplitudes from different codes are not
> directly comparable. Ratios between components are. This package follows the
> package's own documented normalisation and says so in the docstring.

### 2b. Savage: a locked fault that creeps at depth

Between earthquakes, the shallow part of a fault is *locked* while the deeper
part slides steadily. Savage & Burford (1973) modelled this with a screw
dislocation and obtained the famous arctangent profile

$$v(x) = \frac{V}{\pi}\arctan\!\left(\frac{x}{D}\right)$$

where $V$ is the long-term slip rate, $D$ the **locking depth**, and $x$ the
perpendicular distance from the fault trace.

* On the fault trace $x = 0$, so $v = 0$: the fault itself does not move.
  (Coseismic slip is completely different -- there the trace *jumps*.)
* Far away $\arctan \to \pm \pi/2$, so $v \to \pm V/2$. The two sides move in
  opposite directions at half the slip rate each, giving a relative velocity of
  exactly $V$. That is the physical meaning of "long-term slip rate".
* At $x = D$ the argument of arctan is 1, so $v = V/4$ **exactly**.

We place a north-striking fault to the east of the grid, so the two sources sit
side by side.

In [ ]:
# --- Mogi source: a deflating magma chamber at 4 km depth ------------------
mogi = syn.random_mogi_source(grid, style="deflation", rng=np.random.default_rng(7))
print("Mogi source :", {k: (round(v, 3) if isinstance(v, float) else v)
                        for k, v in mogi.items()})
# `random_*_source` returns keyword arguments plus a `style` bookkeeping entry.
mogi_params = dict(mogi)
del mogi_params["style"]
u_e_mogi, u_n_mogi, u_u_mogi = syn.mogi_displacement(grid, **mogi_params)
print("peak |u_U|  : {:.4f} m".format(np.nanmax(np.abs(u_u_mogi))))

# --- Savage source: a north-striking fault creeping at depth ----------------
savage = syn.random_savage_source(grid, style="north_south", rng=np.random.default_rng(3))
savage["center_x"] = 3000.0          # put the trace 3 km east of the centre
savage["locking_depth"] = 8000.0
savage["slip_rate"] = 0.03
print()
print("Savage source:", {k: (round(v, 4) if isinstance(v, float) else v)
                          for k, v in savage.items()})
savage_params = dict(savage)
del savage_params["style"]
v_e, v_n, v_u = syn.savage_velocity(grid, **savage_params)
print("far-field |v_n| equals V/2 = {:.4f} m/yr".format(savage["slip_rate"] / 2))

# --- Sum the two contributions, keeping the components separate -------------
u_e = u_e_mogi + v_e
u_n = u_n_mogi + v_n
u_u = u_u_mogi          # the Savage model has no vertical motion

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)
for ax, field, name in zip(axes, (u_e, u_n, u_u),
                           ("$u_E$ (east)", "$u_N$ (north)", "$u_U$ (up)")):
    lim = float(np.nanmax(np.abs(field)))
    im = ax.imshow(field * 1e3, origin="upper", extent=grid.extent,
                   cmap=SAKURA_DIVERGING, vmin=-lim * 1e3, vmax=lim * 1e3)
    ax.set_title(name + " [mm]")
    ax.set_xlabel("Easting [km]")
    fig.colorbar(im, ax=ax, shrink=0.85)
axes[0].set_ylabel("Northing [km]")
plt.show()


## 3. Projecting onto the satellite's line of sight

A radar does not measure east, north and up separately -- it measures **one
number**: how much the ground moved **along the direction pointing from the
ground to the satellite**. If $\hat{l} = (l_E, l_N, l_U)$ is that unit vector,

$$d_{\mathrm{LOS}} = l_E u_E + l_N u_N + l_U u_U .$$

Two conventions, stated once and never mixed again:

* $\hat{l}$ points **ground $\rightarrow$ satellite**.
* A **positive** $d_{\mathrm{LOS}}$ means the ground moved **toward** the
  satellite, i.e. the range *decreased*.

To build $\hat{l}$ we need the **incidence angle** (how far the look direction
is tilted from vertical; about $34^\circ$ for Sentinel-1's mid-swath) and the
**look azimuth** (the compass bearing from ground to satellite, clockwise from
North).

A right-looking satellite flying on a heading $h$ looks to the right, so its
ground-to-satellite bearing is $h + 270^\circ$. Sentinel-1's near-polar orbit
gives a heading near $-12.6^\circ$, so the look azimuth is about
$257.4^\circ$ -- almost due west, as expected for a satellite passing
northward on your right.

**The most important consequence:** because $l_U = \cos(\theta)$ is the largest
component (about 0.83 at $34^\circ$), and $l_N \approx 0$ for a
north-south orbit, a single interferogram is mostly sensitive to **vertical
motion**, poorly sensitive to north-south motion, and moderately sensitive to
east-west motion. This is why InSAR is paired with GNSS: the radar simply
cannot see the north-south component well.

### Sign bookkeeping

There is exactly **one** place where the overall sign can be chosen: the
`sign` argument of `displacement_to_phase` / `SarGeometry`. Flipping both this
and the direction of $\hat{l}$ cancels out, which is a classic way to invert a
whole dataset without noticing.

In [ ]:
# --- Build the geometry -----------------------------------------------------
geom = syn.SarGeometry(
    wavelength=syn.wavelength_for("sentinel-1"),
    incidence_deg=34.0,
    heading_deg=-12.6,          # near-polar, ascending
)
print("wavelength        :", geom.wavelength, "m")
print("look azimuth      : {:.2f} deg".format(syn.look_azimuth_from_heading(-12.6)))
print("LOS unit vector   :", np.round(geom.los_vector, 4), "(E, N, U)")
print("one fringe is     : {:.2f} cm of LOS motion".format(100 * 0.5 * geom.wavelength))

# --- Project ----------------------------------------------------------------
d_los = geom.project(u_e, u_n, u_u)
print("d_LOS range       : [{:.4f}, {:.4f}] m".format(np.nanmin(d_los), np.nanmax(d_los)))

# A sanity check you can do in your head: pure uplift, no horizontal motion.
up_only = float(geom.project(0.0, 0.0, 1.0))
print("pure 1 m uplift   : d_LOS = {:.4f} m  (cos 34 deg = {:.4f})".format(
    up_only, np.cos(np.deg2rad(34.0))))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), constrained_layout=True)
axes[0].imshow(d_los * 1e3, origin="upper", extent=grid.extent,
               cmap=SAKURA_DIVERGING)
axes[0].set_title("$d_{LOS}$ [mm]")
for ax, v, name in zip(axes[1:], (u_u, u_e),
                       ("$u_U$ (vertical) [mm]", "$u_E$ (east) [mm]")):
    ax.imshow(v * 1e3, origin="upper", extent=grid.extent, cmap=SAKURA_DIVERGING)
    ax.set_title(name)
for ax in axes:
    ax.set_xlabel("Easting [km]")
axes[0].set_ylabel("Northing [km]")
plt.show()

print("Correlation of d_LOS with u_U : {:.4f}".format(
    float(np.corrcoef(d_los.ravel(), u_u.ravel())[0, 1])))
print("Correlation of d_LOS with u_N : {:.4f}".format(
    float(np.corrcoef(d_los.ravel(), u_n.ravel())[0, 1])))


## 4. From displacement to phase

The interferometric phase is proportional to the line-of-sight displacement:

$$\Phi_{\mathrm{def}} = \frac{4\pi}{\lambda}\, d_{\mathrm{LOS}} .$$

**Where does the 4 come from?** The radar pulses travel out and back, so a
change $\Delta r$ in range changes the travelled distance by $2\Delta r$. One
radian of phase corresponds to a travelled distance of $\lambda/(2\pi)$.
Combining the two gives $2 \cdot 2\pi/\lambda = 4\pi/\lambda$.

Because a fringe is $2\pi$ of phase, one fringe equals

$$\frac{2\pi}{4\pi/\lambda} = \frac{\lambda}{2}$$

of displacement -- half a wavelength, as claimed at the top. For Sentinel-1
($\lambda = 5.55$ cm) that is 2.77 cm.

**Watch the numbers.** A 3 cm uplift at $34^\circ$ incidence gives
$d_{\mathrm{LOS}} \approx 2.5$ cm, which is about $4\pi \cdot 0.025 / 0.0555
\approx 5.6$ radians -- nearly a full cycle. This is why real interferograms
are full of fringes and why unwrapping is hard.

In [ ]:
phase_deformation = geom.phase_from_displacement(u_e, u_n, u_u)

cycles = np.nanmax(np.abs(phase_deformation)) / TWO_PI
print("phase range       : [{:.2f}, {:.2f}] rad".format(
    np.nanmin(phase_deformation), np.nanmax(phase_deformation)))
print("peak              : {:.2f} rad = {:.2f} full cycles".format(
    np.nanmax(np.abs(phase_deformation)), cycles))
print("one fringe = {:.2f} mm of displacement".format(
    1e3 / syn.fringe_per_metre(geom.wavelength)))

fig, ax = plt.subplots(figsize=(5.5, 4.4), constrained_layout=True)
im = ax.imshow(phase_deformation, origin="upper", extent=grid.extent, cmap="twilight")
ax.set_title("continuous deformation phase $\\Phi_{{def}}$ [rad]\n"
             "({:.1f} fringes peak-to-peak)".format(cycles))
ax.set_xlabel("Easting [km]")
ax.set_ylabel("Northing [km]")
fig.colorbar(im, ax=ax, shrink=0.85, label="rad")
plt.show()


## 5. Coherence: how much the pixels agree

**Coherence** $\gamma$ is a number in $[0, 1]$ describing how well the phase of
two acquisitions agrees pixel by pixel.

* $\gamma = 1$: perfect agreement, no noise.
* $\gamma = 0$: the two phases are unrelated; the interferogram is garbage.

Physically, coherence is destroyed by vegetation change, surface moisture,
temporal decorrelation (the longer the time between acquisitions, the worse),
and geometric decorrelation.

### A rule that catches everybody once

When **independent** decorrelation mechanisms act together, the coherences
**multiply**. Two areas that each have $\gamma = 0.6$ combine to
$0.6 \times 0.6 = 0.36$ -- much worse than either alone. Losses compound. The
package therefore *multiplies* maps in `combine_coherence`, and says so
loudly, because the intuition "average them" is wrong.

Below we assemble a deliberately busy, realistic coherence map:

| ingredient | what it represents |
|---|---|
| `coherence_gradient` | a west-to-east increase in quality (e.g. farmland into rock) |
| `gaussian_coherence_patch` | a patch of vegetation or agriculture |
| `fault_zone_coherence` | surface rupture / damage along the fault trace |
| `decorrelation_stripe` | a harvesting or flooding stripe|

In [ ]:
coherence = syn.coherence_gradient(grid, low=0.35, high=0.95, azimuth_deg=0.0)
coherence = syn.combine_coherence(
    coherence,
    syn.gaussian_coherence_patch(grid, center_x=-1500.0, center_y=1200.0,
                                 sigma=900.0, depth=0.5),
    syn.fault_zone_coherence(grid, trace_x=np.array([3000.0, 3000.0]),
                             trace_y=np.array([-5000.0, 5000.0]),
                             width=300.0, coherence=0.15),
    syn.decorrelation_stripe(grid, width=900.0, coherence=0.25,
                             azimuth_deg=0.0, offset=1000.0),
)

print("coherence range : [{:.3f}, {:.3f}]".format(
    np.nanmin(coherence), np.nanmax(coherence)))
for name, value in sorted(syn.COHERENCE_LEVELS.items()):
    print("  {:10s} mean gamma = {:.2f}".format(name, value))

fig, ax = plt.subplots(figsize=(5.5, 4.4), constrained_layout=True)
im = ax.imshow(coherence, origin="upper", extent=grid.extent,
               cmap="magma", vmin=0.0, vmax=1.0)
ax.set_title("synthetic coherence $\\gamma$")
ax.set_xlabel("Easting [km]")
ax.set_ylabel("Northing [km]")
fig.colorbar(im, ax=ax, shrink=0.85, label="$\\gamma$")
plt.show()


## 6. Synthetic phase noise -- the part people get wrong

If you add Gaussian noise of some chosen standard deviation to your phase, you
have invented a dataset that no radar ever produced. Real interferometric phase
noise is **circular** (it lives on a circle, not a line) and its distribution
**depends on coherence and on the number of looks**.

### The model

The clean way to build it: imagine a **master** complex pixel
$z_1$ and a **slave** complex pixel $z_2$, both circular complex Gaussian with
unit power mean, correlated such that

$$\langle z_1 z_2^{*}\rangle = \gamma, \qquad \langle |z_1|^2\rangle = \langle |z_2|^2\rangle = 1 .$$

The interferometric *sample* is the product $z_1 z_2^{*}$, and the measured
phase is its argument. Averaging $L$ **independent looks** (either neighbouring
pixels or a longer dwell) gives the interferogram

$$S = \sum_{\ell=1}^{L} z_{1,\ell}\,\overline{z_{2,\ell}}, \qquad
\psi = \arg S .$$

This is exactly what `syn.phase_noise` does. It is not a heuristic -- it is the
textbook InSAR noise model.

### Two useful facts you can verify numerically

1. **Low coherence biases the phase.** The distribution of $\psi$ is not
   centred on zero and is not symmetric; at low coherence the interferometric
   phase is genuinely a biased estimator of the true phase.
2. **The textbook formula is an asymptotic result, not an identity.** The
   familiar approximation $\sigma_{\psi} \approx \sqrt{(1-\gamma^2)/(2L\gamma^2)}$
   is a *large-$L$* limit. At $L = 1$ it can be wrong by more than a factor of
   two, and it only reaches about 1% accuracy once $L$ is in the tens. We will
   measure that below rather than assume it.

In [ ]:
# --- Marginal noise statistics: how good is the textbook formula? ----------
from parvaneh.synthetic import phase_noise_std

print("How well does sigma = sqrt((1 - g^2) / (2 L g^2)) describe the simulator?")
print()
print("{:>6s} {:>7s} {:>13s} {:>13s} {:>8s}".format(
    "gamma", "looks", "sigma(measured)", "sigma(formula)", "ratio"))
for gamma in (0.4, 0.6, 0.8, 0.95):
    for looks in (1, 4, 16, 64):
        samples = syn.phase_noise(np.full(200_000, gamma), looks=looks,
                                  rng=np.random.default_rng(12345))
        sigma = float(np.std(samples))
        formula = float(phase_noise_std(gamma, looks=looks))
        print("{:6.2f} {:7d} {:13.4f} {:13.4f} {:8.3f}".format(
            gamma, looks, sigma, formula, sigma / formula))

print()
print("Read the ratio column, not the absolute values:")
print("  * at looks = 1 the formula is wrong in *both* directions: it")
print("    underestimates sigma by up to a factor 2.25 at gamma = 0.95, and")
print("    overestimates it by about 11 per cent at gamma = 0.40;")
print("  * the ratio walks towards 1.00 as looks grows, reaching 1.01-1.04")
print("    by looks = 64 for every coherence on the list.")
print("So the formula is an asymptotic large-looks result: useful for comparing")
print("scenes, not an exact description of a single-look interferogram.")
print()
print("At gamma = 0 the phase is uniform on the circle and sigma = pi/sqrt(3) =",
      round(np.pi / np.sqrt(3.0), 6))
print("The formula cannot even be evaluated there -- it divides by gamma^2.")


In [ ]:
# --- The single-look phase density, verified against 2 million samples -----
# NOTE: syn.phase_noise already returns the intereferometric phase in radians,
# so it must NOT be passed through np.angle() -- that would reinterpret a real
# number as a complex one and send every negative value to +pi.
gamma_check = 0.6
rng = np.random.default_rng(20240101)
psi = syn.phase_noise(np.full(2_000_000, gamma_check), looks=1, rng=rng)

edges = np.linspace(-np.pi, np.pi, 61)
centres = 0.5 * (edges[:-1] + edges[1:])
width = edges[1] - edges[0]
empirical = np.histogram(psi, bins=edges, density=True)[0]
theory = syn.single_look_phase_pdf(centres, gamma_check)

print("max |empirical - theory| = {:.5f}  (gamma = {})".format(
    float(np.max(np.abs(empirical - theory))), gamma_check))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8), constrained_layout=True)
axes[0].bar(centres, empirical, width=width, alpha=0.55,
            color="#4c72b0", label="2,000,000 simulated single-look phases")
axes[0].plot(centres, theory, "r-", lw=2.0, label="single-look PDF")
axes[0].set_xlabel("interferometric phase $\\psi$ [rad]")
axes[0].set_ylabel("probability density")
axes[0].set_title("$\\gamma = {}$".format(gamma_check))
axes[0].legend(fontsize=8)

for gamma in (0.2, 0.5, 0.8, 0.95):
    axes[1].plot(centres, syn.single_look_phase_pdf(centres, gamma),
                 label="$\\gamma = {}$".format(gamma))
axes[1].set_xlabel("interferometric phase $\\psi$ [rad]")
axes[1].set_title("density widens and flattens as $\\gamma$ falls")
axes[1].legend(fontsize=8)
plt.show()


In [ ]:
# --- Apply the noise to the real phase field -------------------------------
rng = np.random.default_rng(42)
phase_noise_field = syn.phase_noise(coherence, looks=4, rng=rng)
phase_noisy_unwrapped = phase_deformation + phase_noise_field

fig, axes = plt.subplots(1, 3, figsize=(14, 4.0), constrained_layout=True)
for ax, field, name, cmap in (
        (axes[0], phase_deformation, "clean phase $\\Phi_{def}$", "twilight"),
        (axes[1], phase_noise_field, "noise, looks = 4 [rad]", "RdBu_r"),
        (axes[2], phase_noisy_unwrapped, "noisy unwrapped phase", "twilight")):
    im = ax.imshow(field, origin="upper", extent=grid.extent, cmap=cmap)
    ax.set_title(name)
    ax.set_xlabel("Easting [km]")
    fig.colorbar(im, ax=ax, shrink=0.85)
axes[0].set_ylabel("Northing [km]")
plt.show()

print("noise std over the whole image: {:.3f} rad".format(
    float(np.std(phase_noise_field))))
print("mean coherence in that image  : {:.3f}".format(float(np.nanmean(coherence))))


## 7. Wrapping: where the information goes

A phase can only be measured modulo $2\pi$ -- the radar cannot count how many
whole cycles passed. So the *observation* is

$$\phi_{\mathrm{wrap}} = \operatorname{atan2}\!\big(\sin\Phi,\; \cos\Phi\big)
= \arg\big(e^{i\Phi}\big),$$

computed as `np.angle(np.exp(1j * phase))` so that the result is always inside
$[-\pi, \pi]$.

Now the true phase $\Phi = 4.6$ rad and the true phase $\Phi - 2\pi = -1.68$
rad produce **the same** wrapped value. Wrapping is many-to-one, and that
lost integer is *the entire problem* that phase unwrapping exists to solve.

**One subtlety, worth knowing because it breaks floating-point comparisons.**
The mathematically ideal range is $(-\pi, \pi]$, but floating point cannot
represent $\pi$ exactly. This package uses the complex form unconditionally, so
the attainable range is $[-\pi, \pi]$: `wrap_phase(-pi)` returns exactly
$-\pi$, while `wrap_phase(3*pi)` returns a number one unit in the last place
*below* $+\pi$. Both are legitimate wrappings of the same angle -- they differ
by exactly one whole cycle at the endpoint.

### How bad is it here?

Count the wraps. Each 2D fringe in the image is one lost cycle. The number of
wrapped discontinuities you can see is the number of places where an unwrapper
has a chance to make a mistake.

In [ ]:
phase_wrapped = syn.wrap_phase(phase_noisy_unwrapped)

print("wrapped range   : [{:.6f}, {:.6f}]".format(
    np.nanmin(phase_wrapped), np.nanmax(phase_wrapped)))
print("  pi             =  {:.6f}".format(np.pi))
print("wrap_phase(-pi) =  {:.6f}   <- exactly -pi".format(syn.wrap_phase(-np.pi)))
print("wrap_phase(3*pi)=  {:.16f}   <- one ulp below pi".format(syn.wrap_phase(3 * np.pi)))
print()
print("wrapped dynamic range : {:.3f} rad".format(
    np.nanmax(phase_wrapped) - np.nanmin(phase_wrapped)))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)
im0 = axes[0].imshow(phase_deformation, origin="upper", extent=grid.extent, cmap="twilight")
axes[0].set_title("true continuous phase")
im1 = axes[1].imshow(phase_wrapped, origin="upper", extent=grid.extent,
                     cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[1].set_title("wrapped observation")
for ax in axes:
    ax.set_xlabel("Easting [km]")
axes[0].set_ylabel("Northing [km]")
fig.colorbar(im1, ax=axes, shrink=0.8, label="rad")
plt.show()


## 8. The integer ambiguity -- the answer key

Here is the single most valuable output of the whole generator.

Since $\Phi = \phi_{\mathrm{wrap}} + 2\pi k$ for some **integer** $k$, we can
simply solve for $k$:

$$k = \operatorname{round}\!\left(\frac{\Phi - \phi_{\mathrm{wrap}}}{2\pi}\right).$$

`syn.integer_ambiguity` returns that $k$ array. It is the exact number of
cycles that wrapping removed at each pixel -- the ground truth against which an
unwrapper's output can be scored directly, without any of the ambiguity of
comparing to a "reference" run.

### Rounding, carefully

$\Phi - \phi_{\mathrm{wrap}}$ is always an exact multiple of $2\pi$ *in exact
arithmetic*, but floating point makes it very nearly a half-integer at the
boundaries. The package rounds **half away from zero**, implemented explicitly,
rather than relying on `np.round` -- NumPy's `np.round` uses banker's rounding
(round-half-to-even) and has changed behaviour across versions. A tie-break
that silently depends on your NumPy version is not a benchmark.

Once $k$ is known, reconstruction is exact:

$$\phi_{\mathrm{wrap}} + 2\pi k = \Phi .$$

That is the identity we verify below, and it is the first item in the
specification's validation list.

In [ ]:
k = syn.integer_ambiguity(phase_noisy_unwrapped, phase_wrapped)
reconstructed = syn.unwrap_with_ambiguity(phase_wrapped, k)

print("k range           : [{} .. {}]".format(int(k.min()), int(k.max())))
print("k is integer      :", k.dtype)
print("residual |reconstructed - true| : {:.3e} rad".format(
    float(np.nanmax(np.abs(reconstructed - phase_noisy_unwrapped)))))
print("wrap identity max |angle(exp(i*Phi)) - wrap(Phi)| : {:.3e} rad".format(
    float(np.nanmax(np.abs(syn.wrap_phase(phase_noisy_unwrapped) - phase_wrapped)))))
print()
print("Checking the tie-break convention on hand-picked inputs:")
print("  k([pi, -pi, 3pi, -3pi], [pi, pi, pi, pi]) =",
      syn.integer_ambiguity(np.array([np.pi, -np.pi, 3 * np.pi, -3 * np.pi]),
                            np.array([np.pi, np.pi, np.pi, np.pi])))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)
im0 = axes[0].imshow(k, origin="upper", extent=grid.extent, cmap="viridis")
axes[0].set_title("integer ambiguity $k$  (cycles removed)")
axes[1].imshow(reconstructed - phase_noisy_unwrapped, origin="upper",
               extent=grid.extent, cmap="RdBu_r")
axes[1].set_title("reconstruction residual [rad]\n(should be flat zero)")
for ax in axes:
    ax.set_xlabel("Easting [km]")
axes[0].set_ylabel("Northing [km]")
fig.colorbar(im0, ax=axes[0], shrink=0.85, label="cycles")
plt.show()


## 9. Residues: where unwrapping is genuinely impossible

Residues are the crux of the whole subject, and they are easy to define.

Take any $2\times 2$ block of pixels. Walk a closed loop around its four
corners, summing the **wrapped phase differences** along each edge. Because
each edge difference is only known modulo $2\pi$, the total circulation can be

$$\sum_{\text{edges}} \Delta_{\text{wrap}} = 2\pi r, \qquad r \in \{-1, 0, +1\}.$$

* $r = 0$: the loop is consistent. Three of the four phases determine the
  fourth. Locally, unwrapping is unambiguous.
* $r = +1$ or $r = -1$: **a residue.** The phases are mutually inconsistent --
  no assignment of integers can make all four edges agree. The inconsistency is
  a genuine property of the data (usually noise or genuine aliasing), not an
  artefact of the algorithm.

### Why this matters more than it looks

A residue is not a small local annoyance. If you draw any path that *encloses*
a net non-zero number of residues, the unwrapped phase along that path is
ambiguous: you can add $2\pi$ to everything inside and still satisfy every
wrapped observation. This is the discrete version of the classic result that a
gradient field is a true gradient only if every closed loop has zero
circulation.

So every serious algorithm either (a) avoids looping around residues, connecting
them with **branch cuts**, (b) routes **flows** that balance them, as in the MCF
formulation, or (c) integrates only within residue-free regions. Running a naive
integrator that ignores residues can produce errors that spread across the whole
image.

### The balance theorem

On a closed valid domain, the residues must **sum to zero** -- positive and
negative charges always occur in equal numbers. This follows from the fact that
each internal edge is traversed once in each direction and cancels. The package
exposes that as `residue_balance`, and it is one of the specification's required
validation tests.

For a rectangular sub-grid the imbalance is not zero but is exactly the phase
difference between the two right-hand corners, divided by $2\pi$ -- the
"boundary circulation". `syn.boundary_circulation` computes it, which makes the
theorem checkable rather than merely hopeful.

In [ ]:
residues = syn.residue_map(phase_wrapped)
balance = syn.residue_balance(residues)

print("residue map shape :", residues.shape, " dtype", residues.dtype)
print("residues found    : {}".format(balance["absolute"]))
print("  positive (+1)   : {}".format(balance["positive"]))
print("  negative (-1)   : {}".format(balance["negative"]))
print("net charge        : {}".format(balance["total"]))
print("boundary circulation / 2pi = {:.0f}".format(
    float(syn.boundary_circulation(phase_wrapped) / TWO_PI)))
print()
print("On a *closed* domain the net charge must be exactly zero. This whole image")
print("is not closed -- its outer border is itself a loop -- so we look at an")
print("interior block. The block has to be trimmed far enough that its own border")
print("does not cross a discontinuity line, which would add a spurious charge:")
inner = residues[9:-9, 9:-9]
print("  interior positive / negative : {} / {}".format(
    int(np.count_nonzero(inner > 0)), int(np.count_nonzero(inner < 0))))

fig, ax = plt.subplots(figsize=(6.0, 4.8), constrained_layout=True)
ax.imshow(phase_wrapped, origin="upper", extent=grid.extent,
          cmap="twilight", vmin=-np.pi, vmax=np.pi)
rows, cols = np.nonzero(residues)
ys = grid.y[0] - rows * grid.spacing        # residue (i, j) sits at pixel corner
xs = grid.x[0] + cols * grid.spacing
values = residues[rows, cols]
ax.scatter(xs[values > 0], ys[values > 0], s=42, marker="+", c="red",
           linewidths=1.6, label="residue $r = +1$")
ax.scatter(xs[values < 0], ys[values < 0], s=42, marker="x", c="blue",
           linewidths=1.6, label="residue $r = -1$")
ax.set_title("residues overlay the wrapped phase")
ax.set_xlabel("Easting [km]")
ax.set_ylabel("Northing [km]")
ax.legend(loc="upper right", fontsize=8)
plt.show()

pos_density = float(np.count_nonzero(residues > 0)) / residues.size
print("residue density   : {:.4f} residues per pixel".format(pos_density))


## 10. Masks and no-data: the real world is not rectangular

Real interferograms contain water, layover, shadow, incoherent vegetation and
masked-out regions. Unwrappers must cope with **disconnected valid regions** --
you cannot integrate a phase across a hole.

Two design decisions in this package are worth internalising, because getting
them wrong is a classic source of subtle bugs:

1. **Masked pixels are `NaN`, never `0`.** A `0` means "this pixel measured
   zero phase", which is a perfectly ordinary measurement. `NaN` means "there is
   no measurement here". Conflating the two silently invents data.
2. **No-data is not the same as zero coherence.** A pixel with $\gamma = 0$ is
   a valid, hopeless measurement. `NO_DATA` (which is `NaN`) is the absence of a
   measurement. They behave differently in every downstream step.

Let us punch a circular lake out of the scene -- this is exactly the
`circular no-data hole` case in the specification's mask list -- and then look
at how the residues behave around the hole's rim.

In [ ]:
hole = syn.circular_no_data_mask(grid, center_x=-800.0, center_y=-600.0,
                                 radius=1400.0)
print("hole covers {:.1f}% of the image".format(
    100.0 * np.count_nonzero(hole) / hole.size))

phase_masked = np.where(hole, np.nan, phase_wrapped)
coherence_masked = np.where(hole, syn.NO_DATA, coherence)

# The package's own conventions, demonstrated rather than described.
print("masked phase value is NaN      :", bool(np.isnan(phase_masked[hole][0])))
print("NO_DATA is NaN                 :", bool(np.isnan(syn.NO_DATA)))
print("masked coherence is NaN        :", bool(np.isnan(coherence_masked[hole][0])))

# `mask` is a *valid-data* mask, so the hole is passed as ~hole.
residues_masked = syn.residue_map(phase_masked, mask=~hole)
print("residues before masking / after: {} / {}".format(
    int(np.count_nonzero(residues)), int(np.count_nonzero(residues_masked))))
print("  -> residues on the rim vanish, because the loop is no longer closed")

# The residue map is one pixel narrower than the phase in each direction, so
# the hole has to be trimmed to the same (ny - 1, nx - 1) plaquette grid.
hole_small = hole[:-1, :-1]
residues_shown = np.where(hole_small, np.nan, residues_masked.astype(float))

fig, axes = plt.subplots(1, 3, figsize=(14, 4.0), constrained_layout=True)
for ax, field, name, cmap in (
        (axes[0], phase_masked, "wrapped phase with a no-data lake", "twilight"),
        (axes[1], coherence_masked, "coherence", "magma"),
        (axes[2], residues_shown,
         "residues (outside the hole)", "coolwarm")):
    kw = {} if name == "coherence" else dict(vmin=-np.pi, vmax=np.pi)
    if name == "coherence":
        kw = dict(vmin=0.0, vmax=1.0)
    if "residues" in name:
        kw = dict(vmin=-1.0, vmax=1.0)
    im = ax.imshow(field, origin="upper", extent=grid.extent, cmap=cmap, **kw)
    ax.set_title(name)
    ax.set_xlabel("Easting [km]")
    fig.colorbar(im, ax=ax, shrink=0.85)
axes[0].set_ylabel("Northing [km]")
plt.show()


## 11. Validation: proving the scene is self-consistent

A synthetic benchmark is only useful if it is provably correct. The
specification requires a specific list of checks; here they are, run live.

| # | check | why it matters |
|---|---|---|
| 1 | $\arg(e^{i\Phi}) = \phi_{\mathrm{wrap}}$ | wrapping is self-consistent |
| 2 | $\phi_{\mathrm{wrap}} + 2\pi k = \Phi$ | the ambiguity $k$ is exact |
| 3 | zero deformation $\Rightarrow$ zero phase | no spurious phase from the pipeline |
| 4 | $d_{\mathrm{LOS}} = u_U\cos\theta$ for pure uplift | LOS projection is right |
| 5 | Mogi decays like $r^{-3}$ | the elastic solution is right |
| 6 | Savage is antisymmetric in $x$, and $v(0) = 0$ | the interseismic solution is right |
| 7 | Savage far field is $\pm V/2$ | the slip rate is being interpreted correctly |
| 8 | residues sum to zero on a closed domain | the residue operator is right |

All eight are run live in the next cell.  The model-level ones (4-7) are also
locked down by the repository's unit tests -- `tests/test_synthetic.py`,
`tests/test_savage.py`, `tests/test_noise.py` and `tests/test_coherence.py` --
so a regression here cannot pass silently.  Because every number below comes
from the same `grid`, `geom` and `coherence` objects you built above, passing
these checks certifies *this* scene, not some scene in a paper.

In [ ]:
def report(name, ok, detail=""):
    print("{:52s} {:4s}  {}".format(name, "PASS" if ok else "FAIL", detail))
    return ok

results = []

# 1. Wrapping identity -------------------------------------------------------
worst = float(np.nanmax(np.abs(syn.wrap_phase(phase_noisy_unwrapped) - phase_wrapped)))
results.append(report("1. angle(exp(i*Phi)) == wrapped phase", worst < 1e-12,
                      "max error {:.2e} rad".format(worst)))

# 2. Exact reconstruction ----------------------------------------------------
resid = float(np.nanmax(np.abs(reconstructed - phase_noisy_unwrapped)))
results.append(report("2. wrapped + 2*pi*k == true phase", resid < 1e-9,
                      "max error {:.2e} rad".format(resid)))

# 3. Zero deformation gives exactly zero phase -------------------------------
zero_phase = geom.phase_from_displacement(np.zeros(grid.shape),
                                          np.zeros(grid.shape),
                                          np.zeros(grid.shape))
results.append(report("3. zero deformation -> zero phase",
                      bool(np.all(zero_phase == 0.0)), "max |phase| {:.1e} rad".format(
                          float(np.max(np.abs(zero_phase))))))

# 4. LOS projection reduces to cos(incidence) for pure uplift ----------------
up_los = float(geom.project(0.0, 0.0, 1.0))
expected = np.cos(np.deg2rad(34.0))
results.append(report("4. pure uplift -> d_LOS = u_U * cos(incidence)",
                      abs(up_los - expected) < 1e-12,
                      "{:.10f} vs {:.10f}".format(up_los, expected)))

# 5. Mogi far-field decay is r^-3 --------------------------------------------
# u = C d / (r^2 + d^2)^(3/2).  Both probes must sit in the far field, r >> d,
# for the r^-3 law to hold -- so probe at 10 d and 100 d, not near the source.
depth = 1000.0
point = syn.Grid.centered(nx=1, ny=1, spacing=1.0)
u_10 = float(syn.mogi_displacement(point, 10.0 * depth, 0.0, depth, 1e6)[2][0, 0])
u_100 = float(syn.mogi_displacement(point, 100.0 * depth, 0.0, depth, 1e6)[2][0, 0])
ratio = u_100 / u_10
exact = (101.0 / 10001.0) ** 1.5           # closed form of that same ratio
results.append(report("5. Mogi far field decays as r^-3",
                      abs(ratio - exact) < 1e-12 and abs(ratio / 1e-3 - 1.0) < 0.02,
                      "u(100d)/u(10d) = {:.6e}; 1e-3 is the pure r^-3 limit".format(ratio)))

# 6/7. Savage properties -----------------------------------------------------
lonely = syn.Grid.centered(nx=5, ny=3, spacing=100.0)
_, vn_probe, _ = syn.savage_velocity(lonely, 0.0, 8000.0, 0.03)
_, vn_zero, _ = syn.savage_velocity(syn.Grid.centered(nx=1, ny=1, spacing=1.0),
                                    0.0, 8000.0, 0.03)
_, vn_far, _ = syn.savage_velocity(syn.Grid.centered(nx=7, ny=1, spacing=1e6),
                                   0.0, 8000.0, 0.03)
results.append(report("6. Savage is antisymmetric, and v(0) = 0",
                      bool(np.all(vn_probe == -vn_probe[:, ::-1]))
                      and float(vn_zero[0, 0]) == 0.0,
                      "v(x) == -v(-x) exactly; |v(0)| = {:.1f}".format(
                          abs(float(vn_zero[0, 0])))))

# 7. Savage far field approaches +/- V/2 --------------------------------------
half_slip = 0.03 / 2.0
fraction = float(vn_far[0, -1]) / half_slip
results.append(report("7. Savage far field approaches V/2",
                      fraction < 1.0 and abs(fraction - 1.0) < 5e-3,
                      "v/(V/2) = {:.6f} at x = 375 D".format(fraction)))

# 8. Residue balance on a closed domain --------------------------------------
# The loop must not itself cross a discontinuity line, otherwise its own
# circulation is 2*pi and the enclosed charge is legitimately +/-1.  Trimming
# the residue map by 9 pixels on every side already gives a closed loop with
# zero circulation for this scene.
inner_res = syn.residue_map(phase_wrapped)[9:-9, 9:-9]
n_pos = int(np.count_nonzero(inner_res > 0))
n_neg = int(np.count_nonzero(inner_res < 0))
results.append(report("8. residues balance on a closed domain", n_pos == n_neg,
                      "+1: {}   -1: {}".format(n_pos, n_neg)))

print()
print("{} of {} validation checks passed.".format(sum(results), len(results)))


## 12. What you have, and where to go next

You have built a complete, physically defensible synthetic interferogram in
which **every** quantity an unwrapping algorithm must recover is known exactly:

| output | what it is | ground truth? |
|---|---|---|
| `u_e, u_n, u_u` | ENU ground displacement from closed-form elasticity | yes, analytic |
| `d_los` | projection onto the satellite line of sight | yes, linear algebra |
| `phase_deformation` | continuous deformation phase | yes, $4\pi/\lambda \cdot d$ |
| `phase_noisy_unwrapped` | the same phase plus physical InSAR noise | yes, by construction |
| `phase_wrapped` | the observation an algorithm actually sees | the input |
| `integer_ambiguity` | cycles destroyed by wrapping | yes, exact integers |
| `residue_map` | localised inconsistencies | yes, from the wrapped data |
| `coherence` | quality map | yes, chosen by you |

### The three ideas worth remembering

1. **Coherence multiplies.** Independent decorrelation mechanisms compound
   ($0.6 \times 0.6 = 0.36$). Never average them.
2. **Noise is circular and coherence-dependent.** Gaussian noise on the phase is
   not InSAR noise. The correct model is the averaged product of correlated
   complex Gaussians, and it is biased at low coherence.
3. **Residues are a property of the data, not the algorithm.** They mark places
   where no integer assignment can satisfy all the wrapped observations, and
   their sum over a closed domain is always zero.

### Next steps

* Read the `parvaneh.synthetic` module docstrings -- for example
  [`src/parvaneh/synthetic/__init__.py`](../src/parvaneh/synthetic/__init__.py) --
  for the same material in reference form, including every equation and every
  citation, and [`docs/notebooks.md`](../docs/notebooks.md) for the other
  notebooks.
* Run an unwrapper on `phase_wrapped` and score it with
  `evaluate_unwrapper(phase_wrapped, phase_estimated, truth,
  mask, coherence)` -- the integer-cycle error rate it reports is only
  meaningful because of the `k` array you built in section 8.
* Vary the **difficulty**: raise the noise (`looks=1`), lower the coherence,
  add a large atmospheric ramp, or shrink the locking depth until the Savage
  fringes alias. The generator lets you turn each knob independently, which is
  the entire point of building a scene by hand.
* Look at [`docs/phase_unwrapping_history_and_theory.md`](../docs/phase_unwrapping_history_and_theory.md)
  for where residues, branch cuts and network-flow formulations came from.
